# Teste técnico do TCC — LLM no Colab

Objetivo: verificar se conseguimos obter **p(mesmo produto)** de um LLM open-weight e medir o tempo por par.

**Antes de rodar:** menu *Ambiente de execução → Alterar o tipo de ambiente de execução → T4 GPU*. Depois, *Ambiente de execução → Executar tudo*.

Este teste é de viabilidade técnica. Com 10 pares, os acertos e erros não dizem nada sobre a qualidade do modelo.

In [ ]:
import torch, time, math, statistics, csv, platform
import transformers
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "nenhuma (rodando na CPU)")
print("torch", torch.__version__, "| transformers", transformers.__version__, "| python", platform.python_version())

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODELO = "Qwen/Qwen2.5-3B-Instruct"   # modelo do teste; o modelo final sai da revisão bibliográfica
DEV = "cuda" if torch.cuda.is_available() else "cpu"

t0 = time.perf_counter()
tok = AutoTokenizer.from_pretrained(MODELO)
model = AutoModelForCausalLM.from_pretrained(
    MODELO, torch_dtype=torch.float16 if DEV == "cuda" else torch.float32
).to(DEV).eval()
print(f"Modelo carregado em {time.perf_counter() - t0:.0f}s")

ID_1 = tok.encode("1", add_special_tokens=False)
ID_0 = tok.encode("0", add_special_tokens=False)
assert len(ID_1) == 1 and len(ID_0) == 1, "'1' ou '0' não é um token único neste modelo"
ID_1, ID_0 = ID_1[0], ID_0[0]

In [ ]:
INSTRUCAO = (
    "Você compara descrições de produtos de cupons fiscais de supermercados brasileiros. "
    "Duas descrições são o MESMO produto somente se tiverem a mesma marca, a mesma variante "
    "(tipo, sabor, versão) e o mesmo conteúdo (peso ou volume). "
    "Responda apenas com um dígito: 1 se forem o mesmo produto, 0 se forem produtos diferentes."
)

# (descrição A, descrição B, esperado: 1 = mesmo, 0 = diferente, None = não decidível pelo texto)
PARES = [
    ("FERM.ROYAL", "FERMENTO ROYAL 250G", 1),
    ("FERM PO ROYAL 250G", "FERMENTO EM P ROYAL 250G", 1),
    ("BISC RECH MOCA 140G NESTLE", "BIS REC NESTLE MOCA 140G", 1),
    ("MOCA Biscoito Recheado 140g", "BISCOITO RECH. MOCA 140G", 1),
    ("AG CRYSTAL C/G 1,5L", "AG CRYSTAL S/G 1,5L", 0),
    ("LEITE IN XANDO LIGHT GFA 1LT", "LEITE IN XANDO TIPO A GFA 1LT", 0),
    ("ATUM 88 SOL NAT 140G", "ATUM 88 SOLI 140G", 0),
    ("PE DE MOCA AMENDOIM E LEITE CONDENSADO POPULAR 55G", "LEITE COND MOCA", 0),
    ("LEITE COND. MOCA SEMIDESN. TP 395G", "LEITE COND SEMID MOCA TRAD TP 395G", 1),
    ("MOLHO BILLY E JACK 200G OR", "MOLHO BILLY E JACK 200G TA", None),
]

def avaliar(a, b):
    msgs = [{"role": "system", "content": INSTRUCAO},
            {"role": "user", "content": f"Descrição A: {a}\nDescrição B: {b}\nResposta:"}]
    texto = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    entrada = tok(texto, return_tensors="pt").to(DEV)
    t = time.perf_counter()
    with torch.no_grad():
        logits = model(**entrada).logits[0, -1]
    if DEV == "cuda":
        torch.cuda.synchronize()
    seg = time.perf_counter() - t
    lp = torch.log_softmax(logits.float(), dim=-1)
    l1, l0 = lp[ID_1].item(), lp[ID_0].item()
    p_mesmo = math.exp(l1) / (math.exp(l1) + math.exp(l0))   # probabilidade normalizada entre "1" e "0"
    massa = math.exp(l1) + math.exp(l0)                      # quanto da probabilidade total caiu em "1" ou "0"
    top = tok.decode([int(logits.argmax())])
    return {"p_mesmo": p_mesmo, "massa_1_0": massa, "token_mais_provavel": top, "segundos": seg}

In [ ]:
avaliar("TESTE", "TESTE")   # aquecimento, não entra na medição

REPETICOES = 3
linhas, tempos = [], []
for a, b, esp in PARES:
    for rep in range(1, REPETICOES + 1):
        r = avaliar(a, b)
        tempos.append(r["segundos"])
        linhas.append({"modelo": MODELO, "dispositivo": DEV, "descricao_a": a, "descricao_b": b,
                       "esperado": "" if esp is None else esp, "repeticao": rep, **r})
    u = linhas[-1]
    print(f"[{'?' if esp is None else esp}] p(mesmo)={u['p_mesmo']:.3f}  massa={u['massa_1_0']:.2f}  "
          f"top={u['token_mais_provavel']!r:5} {u['segundos']:.2f}s | {a} × {b}")

med = statistics.median(tempos)
print(f"\nTempo por par ({DEV}): mediana {med:.3f}s | mín {min(tempos):.3f}s | máx {max(tempos):.3f}s")
print(f"Estimativa para 1.500 pares: {med * 1500 / 60:.1f} min")
estavel = all(len({round(l['p_mesmo'], 5) for l in linhas if (l['descricao_a'], l['descricao_b']) == (a, b)}) == 1
              for a, b, _ in PARES)
print("Mesma probabilidade nas repetições:", "sim" if estavel else "NÃO")
print("Massa mínima em '1'/'0':", f"{min(l['massa_1_0'] for l in linhas):.2f}",
      "(perto de 1 = o modelo responde no formato pedido)")

with open("resultado_teste_llm_colab.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(linhas[0].keys()))
    w.writeheader(); w.writerows(linhas)
print("Arquivo salvo: resultado_teste_llm_colab.csv (painel de arquivos, à esquerda)")